# Kaggle ASR Full Workflow (v1 / v2 / v3)
Od nule: setup, full run, export artefakata, dotreniranje.

## Pre-run in Kaggle UI
- Settings -> Accelerator: GPU (T4)
- Settings -> Internet: ON
- Add Input datasets: code dataset + raw dataset.
- Save Version (Commit) is needed only to persist outputs as a new dataset for later sessions; it is not required for a single Run All chain in one session.

In [ ]:
import glob
import os
import shutil
import zipfile
from pathlib import Path

workspace = Path('/kaggle/working/speech_recognation')
workspace.mkdir(parents=True, exist_ok=True)

# Case A: dataset je već raspakovan (npr. /kaggle/input/<dataset>/kaggle/requirements_kaggle.txt)
req_candidates = glob.glob('/kaggle/input/**/kaggle/requirements_kaggle.txt', recursive=True)
if req_candidates:
    project_root = Path(req_candidates[0]).parents[1]
    print('Found extracted project at:', project_root)
    shutil.copytree(project_root, workspace, dirs_exist_ok=True)
    os.chdir(workspace)
else:
    # Case B: dataset sadrži zip (npr. speech_recognation_code_for_kaggle.zip)
    zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if not zip_candidates:
        raise FileNotFoundError(
            'Nisam našao ni raspakovan projekat ni zip u /kaggle/input. Attach code dataset.'
        )

    preferred = [z for z in zip_candidates if 'speech_recognation_code_for_kaggle' in z]
    zip_path = preferred[0] if preferred else zip_candidates[0]
    print('Using zip:', zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(workspace)
    os.chdir(workspace)

assert Path('kaggle/requirements_kaggle.txt').exists(), 'Bootstrap failed: missing kaggle/requirements_kaggle.txt'
print('CWD:', Path.cwd())

In [ ]:
!python -m pip install -U pip
!python -m pip install -r kaggle/requirements_kaggle.txt
!python kaggle/prepare_environment.py

In [ ]:
from pathlib import Path
import glob

# If you know the exact path, set it here; leave None for auto-detection.
RAW_DIR = None
WORK_DIR_V1 = '/kaggle/working/asr_full_run_v1'

EPOCHS_V1 = 8
BATCH_SIZE_V1 = 8
LR_V1 = 2e-6
HOLDOUT_RATIO = 0.15

input_root = Path('/kaggle/input')
assert input_root.exists(), 'Missing /kaggle/input'

print('Datasets visible under /kaggle/input:')
for item in sorted(input_root.iterdir()):
    if item.is_dir():
        print(' -', item.name)

# Search recursively because Kaggle may nest datasets as:
# /kaggle/input/datasets/<owner>/<dataset>/...
recursive_raw = [Path(p) for p in glob.glob('/kaggle/input/**/raw', recursive=True)]

candidates = []
for raw_dir in recursive_raw:
    if raw_dir.is_dir():
        audio_count = sum(
            1
            for p in raw_dir.rglob('*')
            if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
        )
        text_count = sum(1 for p in raw_dir.rglob('*.txt'))
        if audio_count > 0 and text_count > 0:
            candidates.append((raw_dir, audio_count, text_count))

# Fallback: if no /raw directories are detected, scan any folder with both audio + txt
if not candidates:
    for p in [Path(x) for x in glob.glob('/kaggle/input/**', recursive=True)]:
        if not p.is_dir():
            continue
        audio_count = sum(
            1
            for f in p.rglob('*')
            if f.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
        )
        text_count = sum(1 for f in p.rglob('*.txt'))
        if audio_count > 0 and text_count > 0:
            candidates.append((p, audio_count, text_count))

if RAW_DIR is None:
    if not candidates:
        raise FileNotFoundError(
            'Auto-detection failed. Please set RAW_DIR manually to your raw dataset path.'
        )

    # Prefer paths containing speech-recognation-raw
    preferred = [x for x in candidates if 'speech-recognation-raw' in str(x[0])]
    selected_path, audio_count, text_count = (preferred[0] if preferred else candidates[0])
    RAW_DIR = str(selected_path.resolve())
else:
    RAW_DIR = RAW_DIR.strip()
    selected_path = Path(RAW_DIR)
    audio_count = sum(
        1
        for p in selected_path.rglob('*')
        if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
    )
    text_count = sum(1 for p in selected_path.rglob('*.txt'))

assert Path(RAW_DIR).exists(), f'RAW_DIR does not exist: {RAW_DIR}'
print('Selected RAW_DIR:', RAW_DIR)
print(f'Found files under RAW_DIR -> audio: {audio_count}, text: {text_count}')

if audio_count == 0 or text_count == 0:
    raise ValueError(
        f'RAW_DIR is not valid for training (audio={audio_count}, text={text_count}).'
    )

In [ ]:
import subprocess

cmd = [
    'python', 'kaggle/run_full_pipeline.py',
    '--raw_dir', RAW_DIR,
    '--work_dir', WORK_DIR_V1,
    '--epochs', str(EPOCHS_V1),
    '--batch_size', str(BATCH_SIZE_V1),
    '--learning_rate', str(LR_V1),
    '--holdout_ratio', str(HOLDOUT_RATIO),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

summary_v1 = Path(WORK_DIR_V1) / 'metrics' / 'comparison_summary.json'
print(json.dumps(json.loads(summary_v1.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
!python kaggle/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v1 --zip_prefix asr_full_run_v1

In [ ]:
import glob
import subprocess
import json
from pathlib import Path

PREV_RUN_V1 = None  # Optional: '/kaggle/input/<your-v1-output-dataset>/asr_full_run_v1'
WORK_DIR_V2 = '/kaggle/working/asr_full_run_v2'

local_v1 = Path('/kaggle/working/asr_full_run_v1')
if local_v1.exists():
    v1_root = local_v1
else:
    if PREV_RUN_V1:
        v1_root = Path(PREV_RUN_V1)
    else:
        discovered = [Path(p) for p in glob.glob('/kaggle/input/**/asr_full_run_v1', recursive=True)]
        if discovered:
            v1_root = discovered[0]
        else:
            raise FileNotFoundError(
                'v1 artifacts not found. Run the v1 cells first in this session, or set PREV_RUN_V1 to your saved dataset path.'
            )

base_model_v1 = v1_root / 'models' / 'raw_aligned_full' / 'final'
train_data_v1 = v1_root / 'aligned_train'
eval_data_holdout = v1_root / 'aligned_holdout'

if not base_model_v1.exists():
    raise FileNotFoundError(f'Base model path does not exist: {base_model_v1}')
if not train_data_v1.exists():
    raise FileNotFoundError(f'Aligned train path does not exist: {train_data_v1}')
if not eval_data_holdout.exists():
    raise FileNotFoundError(f'Aligned holdout path does not exist: {eval_data_holdout}')

cmd = [
    'python', 'kaggle/continue_training.py',
    '--data_dir', str(train_data_v1),
    '--base_model_dir', str(base_model_v1),
    '--output_dir', WORK_DIR_V2,
    '--epochs', '4',
    '--batch_size', '8',
    '--learning_rate', '1.5e-6',
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

metrics_dir_v2 = Path(WORK_DIR_V2) / 'metrics'
metrics_dir_v2.mkdir(parents=True, exist_ok=True)
v2_final = Path(WORK_DIR_V2) / 'final'
v2_metrics_out = metrics_dir_v2 / 'holdout_eval_metrics.json'

eval_cmd_v2 = [
    'python', 'cli/eval_model.py',
    '--model_dir', str(v2_final),
    '--data_dir', str(eval_data_holdout),
    '--split', 'all',
    '--batch_size', '8',
    '--num_beams', '8',
    '--no_repeat_ngram_size', '10',
    '--repetition_penalty', '5.0',
    '--length_penalty', '1.0',
    '--temperature', '0.0',
    '--max_new_tokens', '128',
    '--max_length', '256',
    '--metrics_out', str(v2_metrics_out),
    '--predictions_out', str(metrics_dir_v2 / 'holdout_eval_predictions.jsonl'),
]
print('RUN:', ' '.join(eval_cmd_v2))
subprocess.run(eval_cmd_v2, check=True)

v2_metrics = json.loads(v2_metrics_out.read_text(encoding='utf-8'))
print('V2 holdout metrics (same holdout as v1):')
print(json.dumps({'wer': v2_metrics['wer'], 'cer': v2_metrics['cer'], 'num_samples': v2_metrics['num_samples']}, ensure_ascii=False, indent=2))

!python kaggle/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v2 --zip_prefix asr_full_run_v2

In [ ]:
import glob
import subprocess
import json
from pathlib import Path

PREV_RUN_V2 = None  # Optional: '/kaggle/input/<your-v2-output-dataset>/asr_full_run_v2'
PREV_RUN_V1 = None  # Optional: '/kaggle/input/<your-v1-output-dataset>/asr_full_run_v1'
WORK_DIR_V3 = '/kaggle/working/asr_full_run_v3'

local_v2 = Path('/kaggle/working/asr_full_run_v2')
if local_v2.exists():
    v2_root = local_v2
else:
    if PREV_RUN_V2:
        v2_root = Path(PREV_RUN_V2)
    else:
        discovered_v2 = [Path(p) for p in glob.glob('/kaggle/input/**/asr_full_run_v2', recursive=True)]
        if discovered_v2:
            v2_root = discovered_v2[0]
        else:
            raise FileNotFoundError(
                'v2 artifacts not found. Run the v2 cell first in this session, or set PREV_RUN_V2 to your saved dataset path.'
            )

local_v1 = Path('/kaggle/working/asr_full_run_v1')
if local_v1.exists():
    v1_root = local_v1
else:
    if PREV_RUN_V1:
        v1_root = Path(PREV_RUN_V1)
    else:
        discovered_v1 = [Path(p) for p in glob.glob('/kaggle/input/**/asr_full_run_v1', recursive=True)]
        if discovered_v1:
            v1_root = discovered_v1[0]
        else:
            raise FileNotFoundError(
                'v1 aligned_train not found. Run v1 cells first in this session, or set PREV_RUN_V1 to your saved dataset path.'
            )

base_model_v2 = v2_root / 'final'
train_data_v1 = v1_root / 'aligned_train'
eval_data_holdout = v1_root / 'aligned_holdout'

if not base_model_v2.exists():
    raise FileNotFoundError(f'Base model path does not exist: {base_model_v2}')
if not train_data_v1.exists():
    raise FileNotFoundError(f'Aligned train path does not exist: {train_data_v1}')
if not eval_data_holdout.exists():
    raise FileNotFoundError(f'Aligned holdout path does not exist: {eval_data_holdout}')

cmd = [
    'python', 'kaggle/continue_training.py',
    '--data_dir', str(train_data_v1),
    '--base_model_dir', str(base_model_v2),
    '--output_dir', WORK_DIR_V3,
    '--epochs', '4',
    '--batch_size', '8',
    '--learning_rate', '1.0e-6',
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

metrics_dir_v3 = Path(WORK_DIR_V3) / 'metrics'
metrics_dir_v3.mkdir(parents=True, exist_ok=True)
v3_final = Path(WORK_DIR_V3) / 'final'
v3_metrics_out = metrics_dir_v3 / 'holdout_eval_metrics.json'

eval_cmd_v3 = [
    'python', 'cli/eval_model.py',
    '--model_dir', str(v3_final),
    '--data_dir', str(eval_data_holdout),
    '--split', 'all',
    '--batch_size', '8',
    '--num_beams', '8',
    '--no_repeat_ngram_size', '10',
    '--repetition_penalty', '5.0',
    '--length_penalty', '1.0',
    '--temperature', '0.0',
    '--max_new_tokens', '128',
    '--max_length', '256',
    '--metrics_out', str(v3_metrics_out),
    '--predictions_out', str(metrics_dir_v3 / 'holdout_eval_predictions.jsonl'),
]
print('RUN:', ' '.join(eval_cmd_v3))
subprocess.run(eval_cmd_v3, check=True)

v3_metrics = json.loads(v3_metrics_out.read_text(encoding='utf-8'))
print('V3 holdout metrics (same holdout as v1):')
print(json.dumps({'wer': v3_metrics['wer'], 'cer': v3_metrics['cer'], 'num_samples': v3_metrics['num_samples']}, ensure_ascii=False, indent=2))

!python kaggle/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v3 --zip_prefix asr_full_run_v3

In [ ]:
import json
from pathlib import Path

v1_summary_path = Path('/kaggle/working/asr_full_run_v1/metrics/comparison_summary.json')
v2_metrics_path = Path('/kaggle/working/asr_full_run_v2/metrics/holdout_eval_metrics.json')
v3_metrics_path = Path('/kaggle/working/asr_full_run_v3/metrics/holdout_eval_metrics.json')

if not v1_summary_path.exists():
    raise FileNotFoundError(f'Missing: {v1_summary_path}')
if not v2_metrics_path.exists():
    raise FileNotFoundError(f'Missing: {v2_metrics_path}')
if not v3_metrics_path.exists():
    raise FileNotFoundError(f'Missing: {v3_metrics_path}')

v1_summary = json.loads(v1_summary_path.read_text(encoding='utf-8'))
v2_metrics = json.loads(v2_metrics_path.read_text(encoding='utf-8'))
v3_metrics = json.loads(v3_metrics_path.read_text(encoding='utf-8'))

baseline_wer = float(v1_summary['baseline']['wer'])
baseline_cer = float(v1_summary['baseline']['cer'])
v1_wer = float(v1_summary['finetuned']['wer'])
v1_cer = float(v1_summary['finetuned']['cer'])
v2_wer = float(v2_metrics['wer'])
v2_cer = float(v2_metrics['cer'])
v3_wer = float(v3_metrics['wer'])
v3_cer = float(v3_metrics['cer'])

comparison = {
    'baseline': {'wer': baseline_wer, 'cer': baseline_cer},
    'v1': {'wer': v1_wer, 'cer': v1_cer},
    'v2': {'wer': v2_wer, 'cer': v2_cer},
    'v3': {'wer': v3_wer, 'cer': v3_cer},
    'delta_vs_baseline': {
        'v1_wer': baseline_wer - v1_wer,
        'v2_wer': baseline_wer - v2_wer,
        'v3_wer': baseline_wer - v3_wer,
        'v1_cer': baseline_cer - v1_cer,
        'v2_cer': baseline_cer - v2_cer,
        'v3_cer': baseline_cer - v3_cer,
    },
}

comparison_path = Path('/kaggle/working/asr_full_run_comparison_v1_v2_v3.json')
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f'Saved combined comparison: {comparison_path}')